# 0. 匯入參數

In [10]:
import sys
print(sys.executable)

/home/yingtzudai/Multimodal_SPoD_Video/.venv/bin/python


In [11]:
# 若尚未安裝：pip install "scenedetect[opencv]" "transformers>=4.49"
import warnings, random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModel, AutoProcessor
from scenedetect import open_video, SceneManager, ContentDetector

In [12]:
# ── 資料模式 ──
DATA_MODE = "pilot"        # "pilot" / "full"

ROOT      = Path("/home/yingtzudai/Multimodal_SPoD_Video")
SPLIT_CSV = {"pilot": ROOT / "data/split/pilot_posts.csv",
             "full":  ROOT / "data/split/post_split.csv"}[DATA_MODE]
DATA_ROOT = Path("/home/ftp_246/data_5/多模態2026_ig_top200")
VIDEO_DIR = DATA_ROOT / "Video_data_fullest/ig_vedios"          # {username}/{stem}.mp4
FEAT_DIR  = ROOT / "features"

SEED   = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE  = torch.float16 if device.type == "cuda" else torch.float32
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── 鏡頭切分（MM-AU：PySceneDetect）──
SCENE_THRESHOLD = 27.0     # ContentDetector 預設值 ← MM-AU 未給
MIN_SCENE_SEC   = 0.5      # 最短鏡頭長度          ← MM-AU 未給

# ── 影格擷取（MM-AU：4 fps）──
SAMPLE_FPS  = 4
MAX_FRAMES  = 480          # 超過 90 秒的影片改為均勻抽 360 格 ← 自訂
RESIZE_LONG = 512          # 讀進來先把長邊縮到 512，省記憶體（NaFlex 實際只用到約 336px）

# ── SigLIP 2 ──
SIGLIP_MODEL = "google/siglip2-base-patch16-naflex"   # ablation："google/siglip2-base-patch16-224"
MAX_PATCHES  = 256         # NaFlex 序列長度；9:16 約 12×21 個 patch
FRAME_BATCH  = 32
PROC_KW = {"max_num_patches": MAX_PATCHES} if "naflex" in SIGLIP_MODEL else {}

# ── Pooling / 序列 ──
POOLS     = ["mean_shot", "mean_frame", "max"]
MAX_SHOTS = 35             # MM-AU 的視覺序列上限

# ── 存檔 ──
MODEL_TAG = (SIGLIP_MODEL.split("/")[-1]
             + (f"_p{MAX_PATCHES}" if PROC_KW else "") + f"_fps{SAMPLE_FPS}")
CACHE_DIR = FEAT_DIR / "video" / "cache" / MODEL_TAG   # 每支影片一個檔，pilot / full 共用
OUT_PT    = FEAT_DIR / "video" / f"{DATA_MODE}_{MODEL_TAG}.pt"
SAVE_FRAME_EMB = True      # 保留逐格 embedding，之後換 pooling 不用重算
RECOMPUTE      = False
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(device, "|", MODEL_TAG)

cuda | siglip2-base-patch16-naflex_p256_fps4


# 1. 貼文清單與影片路徑

In [13]:
posts = pd.read_csv(SPLIT_CSV, dtype={"media_id": str, "stem": str})
posts["media_id"]   = posts["media_id"].str.strip()
posts["video_path"] = [VIDEO_DIR / u / f"{s}.mp4" for u, s in zip(posts["username"], posts["stem"])]
posts["has_video"]  = posts["video_path"].map(Path.exists)

print(posts.shape, "| 找得到影片的比例：", posts["has_video"].mean().round(3))
print(posts.loc[~posts["has_video"], ["username", "stem"]].head())

(299, 9) | 找得到影片的比例： 1.0
Empty DataFrame
Columns: [username, stem]
Index: []


# 2. 影片格式檢查

In [14]:
def probe(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None
    fps = cap.get(cv2.CAP_PROP_FPS)
    n   = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    w, h = cap.get(cv2.CAP_PROP_FRAME_WIDTH), cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    cap.release()
    return {"fps": fps, "n_frames": n, "width": w, "height": h,
            "duration": n / fps if fps else np.nan}

sample = posts[posts["has_video"]].sample(min(50, posts["has_video"].sum()), random_state=SEED)
info = pd.DataFrame([probe(p) or {} for p in sample["video_path"]])
print(info.describe().T.round(2))
print("直式比例：", (info["height"] > info["width"]).mean().round(3))
print("4 fps 下平均每支影格數：", (info["duration"] * SAMPLE_FPS).mean().round(0))

          count     mean     std     min      25%      50%      75%      max
fps        50.0    29.76    1.19   23.98    30.00    30.00    30.00    30.00
n_frames   50.0  1476.32  886.87  261.00   725.25  1301.00  2115.50  3610.00
width      50.0   716.80  108.07  360.00   720.00   720.00   720.00  1280.00
height     50.0  1232.00  164.83  640.00  1280.00  1280.00  1280.00  1280.00
duration   50.0    49.45   29.44    8.70    24.18    44.66    70.52   120.33
直式比例： 0.96
4 fps 下平均每支影格數： 198.0


In [15]:
# 全部影片都讀一次檔頭，結果併回 posts
info_all = pd.DataFrame([probe(p) if ok else {} for p, ok in zip(posts["video_path"], posts["has_video"])],
                        index=posts.index)
posts = posts.join(info_all)
posts["orient"] = np.where(posts["height"] > posts["width"], "portrait",
                  np.where(posts["height"] < posts["width"], "landscape", "square"))
posts["dur_bin"] = pd.cut(posts["duration"], [0, 15, 30, 60, 90, np.inf],
                          labels=["<15s", "15-30s", "30-60s", "60-90s", ">90s"])

print("讀不到 / 長度 < 1 秒：", int(((posts["duration"] < 1) | posts["duration"].isna()).sum()), "支")
print("\n各方向的業配比例")
print(posts.groupby("orient")["label"].agg(["mean", "count"]).round(3))
print("\n各長度區間的業配比例")
print(posts.groupby("dur_bin", observed=True)["label"].agg(["mean", "count"]).round(3))
print("\n超過 MAX_FRAMES 上限、會改為均勻抽格的比例：",
      (posts["duration"] * SAMPLE_FPS > MAX_FRAMES).mean().round(3))

讀不到 / 長度 < 1 秒： 0 支

各方向的業配比例
            mean  count
orient                 
landscape  0.600      5
portrait   0.454    293
square     0.000      1

各長度區間的業配比例
          mean  count
dur_bin              
<15s     0.211     38
15-30s   0.227     44
30-60s   0.419    124
60-90s   0.750     64
>90s     0.621     29

超過 MAX_FRAMES 上限、會改為均勻抽格的比例： 0.047


# 3. 鏡頭切分(PySceneDetect)

In [16]:
def detect_shots(path, fps, duration):
    video = open_video(str(path))
    sm = SceneManager()
    sm.add_detector(ContentDetector(threshold=SCENE_THRESHOLD,
                                    min_scene_len=max(1, int(MIN_SCENE_SEC * fps))))
    sm.detect_scenes(video, show_progress=False)
    scenes = sm.get_scene_list(start_in_scene=True)
    shots = [(s.get_seconds(), e.get_seconds()) for s, e in scenes]
    return shots or [(0.0, duration)]          # 一鏡到底 → 整支當一個鏡頭

# 抽 3 支看切得合不合理（可對照影片手動檢查）
for p in sample["video_path"].head(3):
    i = probe(p)
    s = detect_shots(p, i["fps"], i["duration"])
    print(p.name, f"{i['duration']:.1f}s →", len(s), "個鏡頭", [round(a, 1) for a, _ in s][:10])

/tmp/ipykernel_2365750/1479378777.py:8: DeprecationWarning: get_seconds() is deprecated, use the `seconds` property instead.
  shots = [(s.get_seconds(), e.get_seconds()) for s, e in scenes]


thetiffanychen-20251228211256-1216903180549089.mp4 94.6s → 50 個鏡頭 [0.0, 5.4, 6.6, 7.3, 7.9, 8.6, 13.4, 15.4, 16.9, 18.7]
1_shiuan_0-20250908004431-1635094370996640.mp4 51.7s → 13 個鏡頭 [0.0, 10.3, 16.4, 18.1, 34.6, 35.9, 38.6, 44.3, 46.5, 47.7]
zoebitalk-20250630230332-1492024018951094.mp4 13.8s → 11 個鏡頭 [0.0, 5.6, 6.4, 7.1, 7.7, 8.5, 9.4, 10.1, 10.7, 11.8]


# 4. 影格擷取(4 fps)

In [17]:
def sample_frames(path, fps, n_frames):
    duration = n_frames / fps
    times = np.arange(0, duration, 1 / SAMPLE_FPS)
    if len(times) > MAX_FRAMES:
        times = np.linspace(0, duration, MAX_FRAMES, endpoint=False)
    targets = set(np.minimum(np.round(times * fps).astype(int), int(n_frames) - 1).tolist())
    last = max(targets)

    cap, frames, ts, i = cv2.VideoCapture(str(path)), [], [], 0
    while i <= last and cap.grab():
        if i in targets:
            ok, img = cap.retrieve()
            if ok:
                h, w = img.shape[:2]
                s = RESIZE_LONG / max(h, w)
                if s < 1:
                    img = cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA)
                frames.append(Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)))
                ts.append(i / fps)
        i += 1
    cap.release()
    return frames, np.array(ts)

# 5. SigLIP 2 影格 embedding

In [18]:
processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
siglip = AutoModel.from_pretrained(SIGLIP_MODEL, torch_dtype=DTYPE).to(device).eval()

@torch.no_grad()
def encode_frames(frames):
    out = []
    for i in range(0, len(frames), FRAME_BATCH):
        inputs = processor(images=frames[i:i + FRAME_BATCH], return_tensors="pt", **PROC_KW).to(device)
        inputs["pixel_values"] = inputs["pixel_values"].to(DTYPE)
        e = siglip.get_image_features(**inputs)
        if not torch.is_tensor(e):              # 新版 transformers 可能回傳 output 物件
            e = e.pooler_output
        out.append(F.normalize(e.float(), dim=-1).cpu())
    return torch.cat(out)                       # (影格數, d)；base = 768

EMB_DIM = siglip.config.vision_config.hidden_size
print("EMB_DIM:", EMB_DIM)

preprocessor_config.json:   0%|          | 0.00/393 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/329 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/40.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

EMB_DIM: 768


# 6. Pooling: 影格 > 鏡頭 > 影片

In [19]:
def pool_video(frame_emb, ts, shots):
    shot_embs = []
    for s, e in shots:
        m = torch.from_numpy((ts >= s) & (ts < e))
        if m.any():                             # 太短、沒抽到影格的鏡頭跳過
            shot_embs.append(frame_emb[m].mean(0))
    S = torch.stack(shot_embs) if shot_embs else frame_emb.mean(0, keepdim=True)
    return {
        "shots":      S,                        # (鏡頭數, d)：MM-AU 的 shot representation
        "mean_shot":  S.mean(0),                # 每個鏡頭等權（預設）
        "mean_frame": frame_emb.mean(0),        # 每格等權 = 依時長加權
        "max":        frame_emb.max(0).values,  # 對照 SPoD 的 image max-pool
    }

# 7. 批次執行 (可中斷續跑)

In [20]:
def process_video(row):
    out = CACHE_DIR / f"{row.media_id}.pt"
    if out.exists() and not RECOMPUTE:
        return "cached"
    if not Path(row.video_path).exists():
        return "missing"
    info = probe(row.video_path)
    if info is None or not info["fps"] or info["n_frames"] < 1:
        return "unreadable"
    shots = detect_shots(row.video_path, info["fps"], info["duration"])
    frames, ts = sample_frames(row.video_path, info["fps"], info["n_frames"])
    if not frames:
        return "no_frames"
    emb = encode_frames(frames)
    # 幾乎靜止（相鄰影格平均 cosine > 0.99，例如把單張圖片做成影片）→ 一樣存檔，但做標記
    static = len(emb) > 1 and float((emb[1:] * emb[:-1]).sum(-1).mean()) > 0.99
    pooled = pool_video(emb, ts, shots)
    torch.save({"media_id": row.media_id, "model": SIGLIP_MODEL, "info": info,
                "ts": ts, "shots_sec": shots,
                "frame_emb": emb.half() if SAVE_FRAME_EMB else None,
                **{k: v.half() for k, v in pooled.items()}}, out)
    return "ok"

log = []
for row in tqdm(posts.itertuples(), total=len(posts), desc="video"):
    try:
        status = process_video(row)
    except Exception as ex:
        status = f"error: {type(ex).__name__}: {ex}"
    log.append((row.media_id, status))

log = pd.DataFrame(log, columns=["media_id", "status"])
print(log["status"].str.split(":").str[0].value_counts())
log.to_csv(CACHE_DIR.parent / f"{DATA_MODE}_{MODEL_TAG}_log.csv", index=False)

video:   0%|          | 0/299 [00:00<?, ?it/s]

/tmp/ipykernel_2365750/1479378777.py:8: DeprecationWarning: get_seconds() is deprecated, use the `seconds` property instead.
  shots = [(s.get_seconds(), e.get_seconds()) for s, e in scenes]


status
ok    299
Name: count, dtype: int64


# 8. 彙整存檔

In [21]:
N = len(posts)
X_V       = {p: torch.zeros(N, EMB_DIM) for p in POOLS}
shots_pad = torch.zeros(N, MAX_SHOTS, EMB_DIM, dtype=torch.float16)
shot_mask = torch.zeros(N, MAX_SHOTS, dtype=torch.bool)
video_ok  = torch.zeros(N, dtype=torch.bool)
n_shots   = np.zeros(N, dtype=int)

for i, m in enumerate(posts["media_id"]):
    f = CACHE_DIR / f"{m}.pt"
    if not f.exists():
        continue
    c = torch.load(f, weights_only=False)
    for p in POOLS:
        X_V[p][i] = c[p].float()
    S = c["shots"][:MAX_SHOTS]
    shots_pad[i, :len(S)] = S
    shot_mask[i, :len(S)] = True
    n_shots[i] = len(c["shots"])
    video_ok[i] = True

torch.save({"media_id": posts["media_id"].tolist(), "X_V": X_V,
            "shots": shots_pad, "shot_mask": shot_mask, "video_ok": video_ok,
            "config": {"model": SIGLIP_MODEL, "max_patches": MAX_PATCHES, "fps": SAMPLE_FPS,
                       "scene_threshold": SCENE_THRESHOLD, "min_scene_sec": MIN_SCENE_SEC,
                       "max_shots": MAX_SHOTS}}, OUT_PT)

print("已存檔：", OUT_PT)
print("成功編碼：", int(video_ok.sum()), "/", N)
print("鏡頭數分佈：", pd.Series(n_shots[video_ok.numpy()]).describe().round(1).to_dict())
print("超過 35 個鏡頭（會被截斷）的比例：", (n_shots > MAX_SHOTS).mean().round(3))

已存檔： /home/yingtzudai/Multimodal_SPoD_Video/features/video/pilot_siglip2-base-patch16-naflex_p256_fps4.pt
成功編碼： 299 / 299
鏡頭數分佈： {'count': 299.0, 'mean': 17.1, 'std': 16.2, 'min': 1.0, '25%': 4.0, '50%': 13.0, '75%': 25.0, 'max': 122.0}
超過 35 個鏡頭（會被截斷）的比例： 0.117


# 9. 快速檢查：影片向量有沒有業配訊號？

是送進 SPoD 之前的粗略檢查。用 logistic regression 做 5-fold 交叉驗證，並且按網紅分組，同一位網紅的影片不會同時出現在訓練和測試裡。不分組的話，模型可能只是認出「這是某位常接業配的網紅的畫面風格」，分數會被高估。

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.metrics import average_precision_score

ok = video_ok.numpy()
yv, g = posts["label"].values[ok], posts["username"].values[ok]
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
print("隨機基準 AP ≈", yv.mean().round(3))
for p in POOLS:
    prob = cross_val_predict(LogisticRegression(max_iter=2000), X_V[p][ok].numpy(), yv,
                             cv=cv, groups=g, method="predict_proba")[:, 1]
    print(f"{p:10s} AP = {average_precision_score(yv, prob):.3f}")

隨機基準 AP ≈ 0.455
mean_shot  AP = 0.693
mean_frame AP = 0.686
max        AP = 0.775
